# 001 perceptron:
exploring the basics of the perceptron.

following along with: https://medium.com/@becaye-balde/perceptron-building-it-from-scratch-in-python-15716806ef64

```text
Inputs                    Computation                 Output
──────────────────────────────────────────────────────────────

bias = 1 ────────► w₀ ──┐
x₁ ──────────────► w₁ ──┤
x₂ ──────────────► w₂ ──┤
  ⋮                     ├──► weighted_sum ──► activation ──► y
xₙ₋₁ ────────────► wₙ₋₁ ─┤
xₙ ─────────────► wₙ ────┘


weighted_sum = bias·w₀ + x₁·w₁ + x₂·w₂ + ... + xₙ·wₙ

y = activation(weighted_sum)
```

In [ ]:
# import dependencies
import numpy as np

In [ ]:
# perceptron class
class Perceptron(object):
    def __init__(self, eta=0.01, n_iter=10):
        # eta = learning rate is usually a small value between 0.0 and 1.0
        # n_iter = number of iterations.
        # w = weights that define the importance of input value.
        self.eta = eta
        self.n_iter = n_iter
    def weighted_sum(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
    
    def predict(self, X):
        return np.where(self.weighted_sum(X) >= 0.0, 1, -1)
    
    def fit(self, X, y):
        # initializing the weights to 0
        self.w_ = np.zeros(1 + X.shape[1])
        self.errors_ = []
        print("Weights:", self.w_)

        # training the model n_iter times
        for _ in range(self.n_iter):
            error = 0
            
            # CHANGE 1: Use 'target' instead of 'y' for the single label
            for xi, target in zip(X, y):
                
                # 1. calculate ŷ (the predicted value)
                y_pred = self.predict(xi)
                
                # CHANGE 2: Use 'target' instead of 'y' here
                update = self.eta * (target - y_pred)
                
                # 3. Update the weights
                self.w_[1:] = self.w_[1:] + update * xi
                
                # Update the bias (Xo = 1)
                self.w_[0] = self.w_[0] + update
                
                # if update != 0, it means that ŷ != y
                error += int(update != 0.0)
                
            self.errors_.append(error)
        return self

# Maths

Δ(Wi) = η (y — ŷ) * Xi

When y is equal to ŷ, there is no update:
- Δ(W1) = 0.01 * ( 1 − 1 ) ∗ 0.5 = 0
- Δ(W2) = 0.01 * ( (−1) − (−1) ) ∗ 1 = 0

When y is different than ŷ, it means that there’s an error and the weights must be updated by Δ(Wi):
- Δ(W3) = 0.01 * ( 1 − (−1) ) ∗ 4 = 0.08
- Δ(W4) = 0.01 * ( (−1) − 1) ∗ 2 = −0.04

```text
| η | y | ŷ | Xi |
| :---: | :---: | :---: | :---: |
| 0.01 | 1 | 1 | 0.5 |
| 0.01 | -1 | -1 | 1 |
| 0.01 | 1 | -1 | 4 |
| 0.01 | -1 | 1 | 2 |
```

In [4]:
# testing our perceptron
import pandas as pd
from sklearn.utils import shuffle
import numpy as np

df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data', header=None)
# Shuffling the data
df = shuffle(df)

df.head()

,0,1,2,3,4
105,7.6,3.0,6.6,2.1,Iris-virginica
47,4.6,3.2,1.4,0.2,Iris-setosa
35,5.0,3.2,1.2,0.2,Iris-setosa
91,6.1,3.0,4.6,1.4,Iris-versicolor
49,5.0,3.3,1.4,0.2,Iris-setosa


In [ ]:
# Separating the data (X) from the labels (y)
X = df.iloc[:, 0:4].values
y = df.iloc[:, 4].values

print(X[0:5])
print(y[0:5])

[[7.6 3.  6.6 2.1]
 [4.6 3.2 1.4 0.2]
 [5.  3.2 1.2 0.2]
 [6.1 3.  4.6 1.4]
 [5.  3.3 1.4 0.2]]
<StringArray>
[ 'Iris-virginica',     'Iris-setosa',     'Iris-setosa', 'Iris-versicolor',
     'Iris-setosa']
Length: 5, dtype: str


# Splitting the data into train and test set

In [7]:
from sklearn.model_selection import train_test_split

# 75% for train and 25% for test
train_data, test_data, train_labels, test_labels = train_test_split(
                            X, y, test_size=0.25)

# Encoding the labels: 1 for setosa, -1 otherwise
train_labels = np.where(train_labels == 'Iris-setosa', 1, -1)
test_labels = np.where(test_labels == 'Iris-setosa', 1, -1)

print('Train data:', train_data[0:2])
print('Train labels:', train_labels[0:5])

print('Test data:', test_data[0:2])
print('Test labels:', test_labels[0:5])


Train data: [[5.4 3.9 1.7 0.4]
 [6.  2.9 4.5 1.5]]
Train labels: [ 1 -1 -1  1  1]
Test data: [[7.7 3.  6.1 2.3]
 [6.9 3.1 5.4 2.1]]
Test labels: [-1 -1 -1 -1 -1]


In [13]:
# Training the perceptron
perceptron = Perceptron(eta=0.1, n_iter=10)
perceptron.fit(train_data, train_labels)

Weights: [0. 0. 0. 0. 0.]


In [14]:
# Making predictions on test data
test_preds = perceptron.predict(test_data)
print(test_preds)

[-1 -1 -1 -1 -1 -1  1  1 -1 -1 -1  1  1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1  1 -1 -1 -1  1 -1 -1  1 -1  1 -1 -1  1]


In [ ]:
# Measuring Performances
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_preds, test_labels)
print('Accuracy:', round(accuracy, 2) * 100, "%")



# We need a restart

# 001 perceptron:
exploring the basics of the perceptron.

Referance:
https://pabloinsente.github.io/the-perceptron